# Reproduce: When Better Agent Behavior Does Not Improve Accuracy

**Author:** Moran Zavdi · **License:** Apache-2.0

Pins every table / CI claim in [`PAPER.md`](https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition/blob/main/PAPER.md) §§4–8 against compact `evidence/*.jsonl` (**1,350** episodes).

| Resource | Link |
| --- | --- |
| Kaggle competition | https://www.kaggle.com/competitions/gemma-4-developer-agent |
| GitHub | https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition |
| Pages article | https://aeterna-data-ai-ml-research.github.io/Kaggle-Gemma-4-Developer-Agent-Competition/ |
| This dataset | `moranzavdi/gemma4-agent-behavior-vs-accuracy` |

CPU-only; target **&lt; 5 s**. The gate exits non-zero if any pinned claim drifts.


In [ ]:
from pathlib import Path
import subprocess, sys, tarfile

CANDIDATES = [
    Path('/kaggle/input/gemma4-agent-behavior-vs-accuracy'),
    Path('/kaggle/input/datasets/moranzavdi/gemma4-agent-behavior-vs-accuracy'),
    Path('.').resolve(),
    Path('..').resolve(),
]

def has_evidence(p: Path) -> bool:
    return (p / 'evidence' / 'phase1_calibration_900.jsonl').exists()

ROOT = next((p for p in CANDIDATES if has_evidence(p)), None)
if ROOT is None:
    # Kaggle sometimes ships directories as *.tar next to top-level files
    for base in CANDIDATES:
        if not base.exists():
            continue
        work = Path('/kaggle/working/gemma4_unpack')
        work.mkdir(parents=True, exist_ok=True)
        for tar_path in base.glob('*.tar'):
            with tarfile.open(tar_path) as tf:
                tf.extractall(work)
        if has_evidence(work):
            ROOT = work
            break
        if has_evidence(base):
            ROOT = base
            break
assert ROOT is not None, 'Attach dataset moranzavdi/gemma4-agent-behavior-vs-accuracy or clone the GitHub repo'
print('ROOT =', ROOT)

script = ROOT / 'scripts' / 'reproduce_all_tables.py'
if not script.exists():
    # scripts may still be packed
    for tar_path in Path('/kaggle/input/gemma4-agent-behavior-vs-accuracy').glob('scripts.tar') if Path('/kaggle/input/gemma4-agent-behavior-vs-accuracy').exists() else []:
        with tarfile.open(tar_path) as tf:
            tf.extractall(ROOT)
assert script.exists(), script
r = subprocess.run([sys.executable, str(script)], cwd=str(ROOT), capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
assert r.returncode == 0, f'reproduce_all_tables failed: {r.returncode}'
print('ALL_PINNED_CLAIMS_PASS')


## What just passed

- Phase 1: D69 / tr20 / scale10 WT ≈ **30.3–32.0%** (every paired bootstrap CI includes 0)
- Phase 3 Think-1024: WT **48/150 = 48/150**, flips **+9 / −9**
- Oracle pass@6 ≈ **44%** WT
- Counts: 900 + 300 + 150 = **1,350** episodes

Next: open `02_explore_calibration.ipynb` for episode-level EDA.
